# Lab 3 — Build the Pipeline, Hunt the Leaks
**SDA-AIE-111 · Day 2 · Hours 4–5 · 50 min lab + 50 min workshop · pairs**

**Objective.** Deliver `build_preprocessor()` + the full churn pipeline; engineer RFM and calendar features that lift PR-AUC; find all three leaks in `sabotaged_features.ipynb`; pass the shuffled-label test.

**The lesson of the day:** *features beat models.* The same logistic regression, fed order-history features it could not see before, gains more PR-AUC than any model swap on Day 3 will.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               BASE_NUM_COLS, MODEL_NUM_COLS, CAT_COLS, LEAKY)

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
snapshot = pd.Timestamp("2025-11-01")
print(customers.shape, orders.shape)

## Task 1 — The pipeline changes packaging, not math *(10 min)*

Wrap yesterday's logistic regression in `Pipeline(preprocessor → model)`. The score should match Lab 2b within noise — same features, same model, but now **all preprocessing parameters are learned inside the object**, from training data only.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

# TODO: split as in Lab 2 (same seeds), then build
#       Pipeline([("prep", build_preprocessor(BASE_NUM_COLS, CAT_COLS)), ("clf", LogisticRegression(...))])
# TODO: fit on X_tr, score PR-AUC on X_val — confirm Lab 2b parity (packaging, not math)


## Task 2 — RFM aggregates with the snapshot filter *(15 min)*

Open `src/manafeth/features.py` and read `rfm_features()` — the load-bearing line is the snapshot filter. Then build the full feature table and refit.

**The prediction-time rule for aggregates:** an aggregate over all orders summarises the future. Only orders **strictly before the snapshot** may enter a feature.


In [ ]:
# TODO: full = build_feature_table(customers, orders, snapshot); inspect the new columns
# TODO: re-split (same seeds!) and refit the pipeline with
#       BASE_NUM_COLS + ["orders_90d", "basket_trend"]
# TODO: record the PR-AUC lift on the scoreboard


## Task 3 — KSA calendar & behavioural-share features *(10 min)*

The features generic tutorials never have: the KSA weekend is **Fri/Sat**, and Ramadan drives real behaviour. Verify the weekend definition with a spot check on a known Friday, then add `ramadan_order_share` / `weekend_order_share` and `days_to_ramadan`.


In [ ]:
# TODO: spot-check the KSA weekend definition on a known Friday (2025-10-31)
# TODO: refit with MODEL_NUM_COLS (adds ramadan_order_share, weekend_order_share,
#       days_to_ramadan) and record the lift


## Task 4 — Leak hunt *(10 min)*

Open `notebooks/day2/sabotaged_features.ipynb`. It contains **three planted leaks**. Find each one, document it below with the one-line fix, then come back.

| # | Leak | One-line fix |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |


## Task 5 — The shuffled-label test *(5 min)*

If shuffled labels score above chance, something leaks. Run it on your final pipeline; the output goes in your commit message.


In [ ]:
from sklearn.model_selection import cross_val_score

# TODO: permute y_train (seeded), cross_val_score the FULL pipeline against the
#       shuffled labels, and interpret the number: honest ≈ base rate (0.14)


## Workshop hour (Day 2 H5) — propose one feature

Each pair proposes **one new domain feature computed from the orders table only**, implements it inside the pipeline, and posts before/after PR-AUC *from the same seed*. The class votes on the most valuable feature per compute cost.

Starter ideas (do better than these): order-frequency ratio `orders_90d / (3 × orders_per_month)` — is the customer slowing down relative to their own history? · category-diversity in the last 90 days · share of express-slot orders · average items per order trend.

**Commit:** `feat(lab3): leakage-proof pipeline + RFM/calendar features — shuffled-label PR-AUC <your number>`


In [ ]:
# Workshop scratchpad — implement your pair's feature here (orders table only),
# add it to the pipeline column list, and post before/after PR-AUC (same seed).
